# 02 — Model Experimentation

**Thesis:** Macroeconomic Factor-Based Dynamic Portfolio Optimization

This notebook covers:
1. Feature engineering validation
2. Multicollinearity analysis (VIF)
3. Model training & time-series cross-validation
4. Model comparison (Ridge, Lasso, ElasticNet, RF, GBR, XGBoost, LightGBM, SVR)
5. Feature importance analysis
6. Prediction quality assessment

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.config import (
    RAW_DIR, PROCESSED_DIR, RESULTS_DIR,
    TICKERS, TICKER_LIST, PREDICTION_HORIZON,
)
from src.data.preprocessor import (
    build_features, compute_vif, remove_high_vif, stationarity_report,
)
from src.models.trainer import (
    MODELS, train_and_evaluate, train_all_models,
    get_feature_importance, save_model,
)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.figsize': (14, 6), 'figure.dpi': 100})

print(f'Available models: {list(MODELS.keys())}')
print(f'Prediction horizon: {PREDICTION_HORIZON} days')

## 1. Load & Validate Features

In [ ]:
# Load features (build if not available)
try:
    features = pd.read_csv(PROCESSED_DIR / 'features.csv', index_col=0, parse_dates=True)
    print(f'Loaded features: {features.shape}')
except FileNotFoundError:
    prices = pd.read_csv(RAW_DIR / 'prices.csv', index_col=0, parse_dates=True)
    macro = pd.read_csv(RAW_DIR / 'macro.csv', index_col=0, parse_dates=True)
    features = build_features(prices, macro)

print(f'Features: {features.shape[1]} columns, {features.shape[0]} rows')
print(f'Date range: {features.index[0].date()} to {features.index[-1].date()}')
print(f'\nNull values: {features.isnull().sum().sum()}')
print(f'Inf values: {np.isinf(features.select_dtypes(include=[np.number])).sum().sum()}')

In [ ]:
# Feature categories
ret_cols = [c for c in features.columns if '_ret_' in c]
vol_cols = [c for c in features.columns if '_vol_' in c]
mom_cols = [c for c in features.columns if '_mom_' in c]
macro_cols = [c for c in features.columns if c not in ret_cols + vol_cols + mom_cols]

print(f'Return features: {len(ret_cols)}')
print(f'Volatility features: {len(vol_cols)}')
print(f'Momentum features: {len(mom_cols)}')
print(f'Macro features: {len(macro_cols)}')

## 2. Multicollinearity Analysis (VIF)

In [ ]:
# Select a subset of features for VIF (too many columns makes VIF slow)
# Use one ticker's features + macro features
sample_ticker = 'SPY'
sample_cols = (
    [c for c in features.columns if c.startswith(f'{sample_ticker}_') and not c.endswith(f'_ret_{PREDICTION_HORIZON}d')]
    + [c for c in macro_cols if 'lag' not in c]  # base macro only
)
sample_cols = [c for c in sample_cols if c in features.columns][:20]  # limit for speed

print(f'Computing VIF for {len(sample_cols)} features...')
vif_df = compute_vif(features[sample_cols].dropna())

fig, ax = plt.subplots(figsize=(12, 6))
vif_plot = vif_df.head(20).iloc[::-1]
colors = ['red' if v > 10 else 'orange' if v > 5 else 'steelblue' for v in vif_plot['vif']]
ax.barh(vif_plot['feature'], vif_plot['vif'], color=colors)
ax.axvline(x=10, color='red', linestyle='--', label='VIF = 10 (high collinearity)')
ax.axvline(x=5, color='orange', linestyle='--', label='VIF = 5 (moderate)')
ax.set_title('Variance Inflation Factors')
ax.set_xlabel('VIF')
ax.legend()
plt.tight_layout()
plt.show()

high_vif = vif_df[vif_df['vif'] > 10]
print(f'\n{len(high_vif)} features with VIF > 10 (multicollinearity concern):')
if len(high_vif) > 0:
    print(high_vif.to_string(index=False))

## 3. Prepare Training Data

In [ ]:
# Prepare X (features) and y (target) for a representative asset
target_ticker = 'SPY'
horizon = PREDICTION_HORIZON
target_col = f'{target_ticker}_ret_{horizon}d'

# Feature columns: everything except return targets at the prediction horizon
feature_cols = [c for c in features.columns if not c.endswith(f'_ret_{horizon}d')]

X = features[feature_cols]
y = features[target_col]

# Clean
mask = X.notna().all(axis=1) & y.notna()
X_clean = X[mask]
y_clean = y[mask]

print(f'Target: {target_col}')
print(f'X shape: {X_clean.shape}')
print(f'y shape: {y_clean.shape}')
print(f'y stats: mean={y_clean.mean():.4f}, std={y_clean.std():.4f}')

## 4. Train All Models — Time-Series CV

In [ ]:
# Train all models with time-series cross-validation
print(f'Training {len(MODELS)} models on {target_ticker} with time-series CV...')
print('=' * 60)

results = train_all_models(X_clean, y_clean)

print('\n' + '=' * 60)
print('MODEL COMPARISON (sorted by RMSE)')
print('=' * 60)
comparison = results['comparison']
display(comparison.style.format({
    'rmse_mean': '{:.6f}', 'rmse_std': '{:.6f}',
    'mae_mean': '{:.6f}', 'r2_mean': '{:.4f}', 'r2_std': '{:.4f}',
}).background_gradient(subset=['rmse_mean'], cmap='RdYlGn_r')
 .background_gradient(subset=['r2_mean'], cmap='RdYlGn'))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE comparison
ax = axes[0]
comp_sorted = comparison.sort_values('rmse_mean')
ax.barh(comp_sorted['model'], comp_sorted['rmse_mean'], xerr=comp_sorted['rmse_std'], 
        color='steelblue', capsize=3)
ax.set_title('RMSE by Model (lower is better)')
ax.set_xlabel('RMSE')

# R² comparison
ax = axes[1]
comp_sorted = comparison.sort_values('r2_mean', ascending=False)
colors = ['green' if r > 0 else 'red' for r in comp_sorted['r2_mean']]
ax.barh(comp_sorted['model'], comp_sorted['r2_mean'], xerr=comp_sorted['r2_std'],
        color=colors, capsize=3)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_title('R² by Model (higher is better)')
ax.set_xlabel('R²')

# MAE comparison
ax = axes[2]
comp_sorted = comparison.sort_values('mae_mean')
ax.barh(comp_sorted['model'], comp_sorted['mae_mean'], color='coral', capsize=3)
ax.set_title('MAE by Model (lower is better)')
ax.set_xlabel('MAE')

plt.suptitle(f'Model Comparison — {target_ticker} {horizon}-Day Return Prediction', fontsize=14)
plt.tight_layout()
plt.show()

## 5. CV Fold-by-Fold Analysis

In [ ]:
# Per-fold performance for top 3 models
top_models = comparison.head(3)['model'].tolist()

fig, axes = plt.subplots(1, len(top_models), figsize=(6 * len(top_models), 5))
if len(top_models) == 1:
    axes = [axes]

for ax, model_name in zip(axes, top_models):
    cv_metrics = results['models'][model_name]['cv_metrics']
    
    x = cv_metrics['fold']
    ax.bar(x - 0.15, cv_metrics['rmse'], 0.3, label='RMSE', color='steelblue')
    ax2 = ax.twinx()
    ax2.bar(x + 0.15, cv_metrics['r2'], 0.3, label='R²', color='coral')
    
    ax.set_title(f'{model_name}')
    ax.set_xlabel('CV Fold')
    ax.set_ylabel('RMSE', color='steelblue')
    ax2.set_ylabel('R²', color='coral')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')

plt.suptitle('Per-Fold CV Performance', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Feature importance from best tree-based model
tree_models = [m for m in top_models if m in ['xgboost', 'lightgbm', 'random_forest', 'gradient_boosting']]
if not tree_models:
    tree_models = ['xgboost']  # fallback

fig, axes = plt.subplots(1, len(tree_models), figsize=(8 * len(tree_models), 8))
if len(tree_models) == 1:
    axes = [axes]

for ax, model_name in zip(axes, tree_models):
    if model_name in results['models']:
        fi = results['models'][model_name].get('feature_importance')
        if fi is not None:
            top_fi = fi.head(20).iloc[::-1]
            ax.barh(top_fi['feature'], top_fi['importance_pct'])
            ax.set_title(f'{model_name} — Top 20 Features')
            ax.set_xlabel('Importance (%)')

plt.suptitle(f'Feature Importance — {target_ticker} {horizon}-Day Return Prediction', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-model feature importance consensus
all_fi = []
for model_name, model_result in results['models'].items():
    fi = model_result.get('feature_importance')
    if fi is not None and not fi['importance'].isna().all():
        fi_norm = fi.copy()
        fi_norm['model'] = model_name
        fi_norm['rank'] = range(1, len(fi_norm) + 1)
        all_fi.append(fi_norm)

if all_fi:
    combined_fi = pd.concat(all_fi, ignore_index=True)
    avg_rank = combined_fi.groupby('feature')['rank'].mean().sort_values()
    
    print('Top 15 Features by Average Rank Across Models:')
    print('=' * 50)
    for i, (feat, rank) in enumerate(avg_rank.head(15).items()):
        print(f'  {i+1:2d}. {feat:40s} avg_rank = {rank:.1f}')

## 7. Train Models for All Assets

In [ ]:
# Train best model type for all tickers
best_model_type = comparison.iloc[0]['model']
print(f'Best model type: {best_model_type}')
print(f'Training for all {len(TICKER_LIST)} assets...\n')

all_asset_results = {}
asset_metrics = []

for ticker in TICKER_LIST:
    target_col = f'{ticker}_ret_{horizon}d'
    if target_col not in features.columns:
        print(f'  Skipping {ticker}: target not found')
        continue
    
    y_t = features[target_col]
    mask = X_clean.index.isin(features.index) & y_t.notna()
    X_t = features.loc[mask, feature_cols]
    y_t = y_t[mask]
    
    if len(X_t) < 500:
        print(f'  Skipping {ticker}: only {len(X_t)} samples')
        continue
    
    result = train_and_evaluate(X_t, y_t, model_name=best_model_type)
    all_asset_results[ticker] = result
    save_model(result, target_col)
    
    cv = result['cv_metrics']
    asset_metrics.append({
        'ticker': ticker,
        'asset_class': TICKERS[ticker],
        'rmse': cv['rmse'].mean(),
        'r2': cv['r2'].mean(),
        'mae': cv['mae'].mean(),
    })
    print(f'  {ticker:5s} ({TICKERS[ticker]:35s}) — RMSE: {cv["rmse"].mean():.6f}, R²: {cv["r2"].mean():.4f}')

asset_metrics_df = pd.DataFrame(asset_metrics)
asset_metrics_df.to_csv(RESULTS_DIR / 'asset_model_metrics.csv', index=False)

In [ ]:
# Per-asset R² visualization
fig, ax = plt.subplots(figsize=(12, 5))
am = asset_metrics_df.sort_values('r2', ascending=True)
colors = ['green' if r > 0 else 'red' for r in am['r2']]
ax.barh(am['ticker'] + ' — ' + am['asset_class'], am['r2'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_title(f'{best_model_type} — Cross-Validated R² by Asset')
ax.set_xlabel('R²')
plt.tight_layout()
plt.show()

print(f'\nAverage R² across assets: {asset_metrics_df["r2"].mean():.4f}')
print(f'Assets with R² > 0: {(asset_metrics_df["r2"] > 0).sum()}/{len(asset_metrics_df)}')

## 8. Key Findings

1. **Model ranking**: Tree-based ensembles (XGBoost, LightGBM, GBR) typically outperform linear models.
2. **R² reality check**: R² for monthly return prediction is low (financial returns are noisy). Even small positive R² is economically meaningful.
3. **Feature importance**: Macro features (VIX, yield spread, sentiment) appear alongside momentum/volatility as top predictors.
4. **Multicollinearity**: Some feature pairs have high VIF — regularized models (Ridge, ElasticNet) handle this well.
5. **Asset variation**: Prediction quality varies by asset class — equities are typically harder than bonds.